# Formative 2, Task 6: System Simulation

**Author:** Bakhit. This notebook demonstrates the full multimodal authentication and product recommendation flow, including authorized and unauthorized attempts.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import sys
import os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), ".."))

from scripts.predict_face import predict_face
from scripts.predict_product import predict_product
from scripts.verify_voice import verify_voice

print("all three models loaded successfully")

## 1. System Flow Overview

The system flow matches the assignment diagram: a user presents a face image to the face recognition gate first, since identity has to be confirmed before anything else runs. If the face is recognized, the system runs the product recommendation model for that user, then asks for a voice sample as a second, independent identity check. Only if voice verification also passes does the system display the recommended product. If either gate fails, access is denied immediately and the flow stops, so a stranger can never reach the product model or the final display just by getting past one check.

In [ ]:
def simulate_transaction(face_path, voice_path, label=""):
    print("=" * 60)
    print(label)
    print("=" * 60)

    print("\nSTEP 1: Face recognition gate")
    face_label, face_confidence = predict_face(face_path)
    print(f"  input: {face_path}")
    print(f"  result: {face_label}, confidence={face_confidence:.3f}")
    if face_label == "unauthorized":
        print("\nACCESS DENIED: face not recognized")
        return {
            "Simulation": label, "Face Input": face_path,
            "Face Result": f"{face_label} ({face_confidence:.3f})",
            "Voice Input": "-", "Voice Result": "-",
            "Product Shown": "-", "Outcome": "DENIED (face)"
        }

    print("\nSTEP 2: Product recommendation")
    sample_features = {
        "engagement_score": 75.0,
        "purchase_interest_score": 3.5,
        "sentiment_score": 0.5,
        "n_platforms": 2,
        "n_transactions": 10,
        "total_spend": 500.0,
        "avg_amount": 50.0,
        "max_amount": 150.0,
        "avg_rating": 4.0,
        "n_categories": 3,
        "spend_per_txn": 50.0,
        "interest_x_engagement": 262.5,
        "is_high_value": 1,
    }
    product = predict_product(sample_features)
    print(f"  recommended product: {product}")

    print("\nSTEP 3: Voice verification gate")
    voice_label, voice_confidence = verify_voice(voice_path)
    print(f"  input: {voice_path}")
    print(f"  result: {voice_label}, confidence={voice_confidence:.3f}")
    if voice_label == "unauthorized":
        print("\nACCESS DENIED: voice not verified")
        return {
            "Simulation": label, "Face Input": face_path,
            "Face Result": f"{face_label} ({face_confidence:.3f})",
            "Voice Input": voice_path,
            "Voice Result": f"{voice_label} ({voice_confidence:.3f})",
            "Product Shown": product, "Outcome": "DENIED (voice)"
        }

    print("\n" + "=" * 60)
    print("TRANSACTION APPROVED")
    print(f"  user: {face_label}")
    print(f"  recommended product: {product}")
    print("=" * 60)
    return {
        "Simulation": label, "Face Input": face_path,
        "Face Result": f"{face_label} ({face_confidence:.3f})",
        "Voice Input": voice_path,
        "Voice Result": f"{voice_label} ({voice_confidence:.3f})",
        "Product Shown": product, "Outcome": "APPROVED"
    }

## 2. Simulation 1: Authorized Transaction

Bakhit presents his own face photo and voice clip. Both should pass the authentication gates, and the system should display a product recommendation. This proves the full end-to-end flow works for a legitimate user.

In [ ]:
result_1 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/Bakhit2.wav",
    label="SIMULATION 1: Authorized Transaction (Bakhit)"
)

**[PLACEHOLDER, fill in after running]** Cite the actual face confidence, the recommended product, and the voice confidence from the output above. Confirm both gates passed and the transaction was approved.

## 3. Simulation 2: Unauthorized Face Attempt

A stranger's photo is presented instead of a team member's. The face gate should reject it immediately, at 0.325 confidence, well under the 0.55 threshold, and the system should never reach the product model or the voice gate. This proves the first security checkpoint works on its own.

In [ ]:
result_2 = simulate_transaction(
    face_path="../images/unauthorized/attempt1.jpg",
    voice_path="../audio/Bakhit2.wav",
    label="SIMULATION 2: Unauthorized Face Attempt"
)

**[PLACEHOLDER, fill in after running]** Cite the actual confidence score from the output above and confirm access was denied at step 1, before the product model or voice gate ever ran.

## 4. Simulation 3: Unauthorized Voice Attempt

Bakhit's own photo passes the face gate, so the flow reaches the product model and the voice gate, but a stranger's voice clip is presented instead of Bakhit's own. The voice gate should reject that clip at 0.340 confidence, under the 0.6 threshold. This proves that passing the face gate alone is not enough: both biometric checks have to succeed before the product is actually shown.

In [ ]:
result_3 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/stranger1.wav",
    label="SIMULATION 3: Unauthorized Voice Attempt"
)

**[PLACEHOLDER, fill in after running]** Cite the face confidence (should pass), the product prediction (should still run), and the voice confidence (should fail) from the output above. Note that the system correctly stopped at the voice gate instead of approving the transaction.

## 5. Simulation 4: Second Unauthorized Voice Attempt

This repeats simulation 3 with the second stranger clip instead of the first, since stranger2.wav is rejected at 0.365 confidence in prior testing, a different number from stranger1's 0.340. Running both shows the voice gate consistently rejects unknown voices, not just one specific recording that happens to sit far below the threshold.

In [ ]:
result_4 = simulate_transaction(
    face_path="../images/Bakhit/neutral.jpg",
    voice_path="../audio/stranger2.wav",
    label="SIMULATION 4: Second Unauthorized Voice Attempt"
)

**[PLACEHOLDER, fill in after running]** Compare stranger2's confidence to stranger1's confidence from simulation 3. Confirm both sit below the 0.6 threshold, even though they are not the same number.

## 6. Summary of All Simulations

In [ ]:
import pandas as pd

summary_df = pd.DataFrame([result_1, result_2, result_3, result_4],
                          columns=["Simulation", "Face Input", "Face Result",
                                   "Voice Input", "Voice Result", "Product Shown", "Outcome"])
display(summary_df)

**[PLACEHOLDER, fill in after running]** Cite the summary table: confirm the authorized transaction was approved, and that both unauthorized attempts were correctly blocked, one at the face gate and two at the voice gate. Explain why the two-factor design matters here: a face-only system would have let both voice strangers through once Bakhit's own photo was shown, and a voice-only system would have let the face stranger through if a valid voice clip had been paired with it.

## 7. System Limitations

- The face model reaches 1.000 accuracy on 4 test photos, but 4 photos is a very small test set, and it may not generalize to photos taken in different lighting or camera angles than the training set.
- The voice model reaches 0.500 accuracy on 4 test clips, with David and Serein consistently misclassified. A production system would need more training clips per member and longer recordings.
- This simulation uses hardcoded sample features for the product model instead of real data from the merged customer dataset. In a real deployment, those features would come from the authenticated user's actual social media profile and transaction history.
- Bakhit is the demo user here because he is the only member reliably recognized by both the face model and the voice model. A production system would need that same reliability across every enrolled member.
- The confidence thresholds used here, 0.55 for face and 0.6 for voice, were tuned to this specific dataset. A different user base or recording environment would need its own tuning.

## References

This notebook uses models trained in the following companion notebooks:

- Task 1: `notebooks/task1_data_merge.ipynb` (product recommendation model, Serein)
- Task 2: `notebooks/B_image_face_model.ipynb` (facial recognition model, David)
- Task 3: `notebooks/C_audio_voice_model.ipynb` (voiceprint verification model, Divine)